## Results require recomputation after the 2026-11-06 feature correction

The price-adjustment timing has been corrected and the feature table rebuilt. Saved outputs were cleared and prior models, metrics and model-selection results were preserved under archive/price_adjustment_fix_20261106. Rerun notebook 03 and training/validation in 04 before interpreting results or using 05. Any previously written numerical conclusions require review. 2021 was evaluated earlier; repeating it is a reevaluation, not a new unseen test.


## HT2026-DA631E-TS056
## Mini Project 1
### Author: Nancy Truong, Argtim Zeqiraj
#### Final test evaluation

Refit the selected model family on all permitted development data, transform the 2021 test set with that same fitted pipeline, evaluate once and save atifacts.

On the held-out 2021 test period, the selected HGB model achieved an **unannualized spread Sharpe of 0.0867**, compared with a mean validation Sharpe of **0.2359**. Performance was therefore weaker on unseen data. The mean daily Spearman IC of **0.003454** indicates only a weak positive association between predicted and actual stock rankings.

Return-prediction accuracy was also limited. The MAE of **0.015337** corresponds to an average absolute error of approximately **1.53 percentage points**. The MSE of **0.000495** corresponds to an RMSE of approximately **2.22 percentage points**. The negative $R^2$ of **−0.0261** means that squared prediction error was approximately **2.61% greater than the test-mean reference used by this metric**.

Performance varied considerably over time. Quarterly spread Sharpe was positive in Q1, Q2, and Q3 (**0.1893, 0.1453, and 0.1516**), but became negative in the available Q4 period (**−0.2038**). Mean daily IC was also negative in October and November, showing that ranking performance deteriorated during these months. December contains only three observations of daily IC, so its slightly positive mean provides little evidence of recovery.

The positive aggregate spread score therefore conceals a late-period reversal. These results suggest weak and unstable predictive performance; they do not establish consistent trading profitability, particularly because transaction costs and execution constraints are not included.

In [ ]:
from pathlib import Path
import sys
import json
import platform
import time
import joblib
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the parent directory containing src/data.py."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data.py").is_file():
            return candidate

    raise FileNotFoundError("Could not find src/data.py")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
    
from config import ProjectConfig, ProjectPaths

config = ProjectConfig()
paths = ProjectPaths(PROJECT_ROOT)
paths.ensure_output_dirs

from modeling import (
    build_model_pipeline
    ,chronological_split
    ,development_and_test_split
    ,make_xy
    ,save_json
    ,select_recent_dates
    ,evaluate_predictions
    ,walk_forward_split
    ,make_fold_metric_record
)
from features import (
    infer_feature_columns
)


print("Project root:", PROJECT_ROOT)
print("Imported load_data successfully")

print("Imported libraries sucessfully! yay")

In [ ]:
selected_path = paths.reports / "selected_model.json"
if not selected_path.is_file():
    raise FileNotFoundError("Run notebook 04 and save the model family first")

selection = json.loads(selected_path.read_text(encoding='utf-8'))
selected_model = selection['selected_model']
FAST_MODE = bool(selection['fast_mode'])
print(f"Selected model: {selected_model}; FAST_MODE={FAST_MODE}")

### Rebuild development data (train + validation) and test sets

Train and validation datasets are now combined becuase the model is fixed. Use the combined set to train and no longer need a purge between train and validation.

The final two development dates are purged because their labels cross into the 2021 test period

In [ ]:
features_table = pd.read_parquet(paths.processed / "features_table.parquet")
numeric_features, categorical_features = infer_feature_columns(features_table)
development, test, purged_dates = development_and_test_split(features_table, config)

split_summary = pd.DataFrame({
    'split': ["development","test"]
    ,'rows': [len(development), len(test)]
    ,'dates': [development['Date'].nunique(), test['Date'].nunique()]
    ,'date_min': [development['Date'].min(), test['Date'].min()]
    ,'date_max': [development['Date'].max(), test['Date'].max()]
}).set_index('split')
display(split_summary)

print("Purged before test: ", [str(date.date()) for date in purged_dates])

### Fit on development and transform test

In [ ]:
X_development, y_development = make_xy(development, numeric_features, categorical_features)
X_test, y_test = make_xy(test, numeric_features, categorical_features)

pipeline = build_model_pipeline(
    model_name = selected_model
    ,numeric_features= numeric_features
    ,categorical_features= categorical_features
    ,random_seed = config.random_seed
    ,fast_mode= FAST_MODE
    ,model_params=selection['params']
    )

started = time.perf_counter()
pipeline.fit(X_development, y_development)
fit_seconds = time.perf_counter() - started

test_predictions = pipeline.predict(X_test)
assert len(test_predictions) == len(test)
assert np.isfinite(test_predictions).all()

print(f"Final fit completed in {fit_seconds:1f} seconds")

### Open test result

Interpret ranking metrics

In [ ]:
test_metrics, test_ranked, test_daily_ic, test_daily_spread_return = evaluate_predictions(
    df = test
    ,predictions = test_predictions
    ,config=config
    ,strict_portfolio_size = True
)
test_metrics.loc['FinalFitSeconds'] = fit_seconds

display(test_metrics.to_frame())

In [ ]:
rank_check = test_ranked.groupby('Date')['Rank'].agg(['min','max','nunique','size'])
assert rank_check['min'].eq(0).all()
assert rank_check['nunique'].eq(rank_check['size']).all()
assert rank_check['max'].eq(rank_check['size'] - 1).all()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
test_daily_ic.rolling(20, min_periods=10).mean().plot(ax=axes[0], color="#4C78A8")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Final test: 20-day rolling mean daily rank IC")
axes[0].set_ylabel("Spearman IC")

test_daily_spread_return.cumsum().plot(ax=axes[1], color="#E45756")
axes[1].set_title("Final test: cumulative JPX-style spread return")
axes[1].set_ylabel("Cumulative spread")
plt.tight_layout()
plt.show()

In [ ]:
monthly_ic = test_daily_ic.groupby(test_daily_ic.index.to_period("M")).agg(["mean", "median", "count"])
quarterly_spread = test_daily_spread_return.groupby(test_daily_spread_return.index.to_period("Q")).agg(
    ["mean", "std", "sum", "count"]
)
quarterly_spread["Sharpe"] = quarterly_spread["mean"] / quarterly_spread["std"]
display(monthly_ic)
display(quarterly_spread)

In [ ]:
def plot_stock_rank_history(
    predictions, security_code=None, *, portfolio_size=200,
    split_label="2021 test",
):
    """Compare one stock's predicted/actual daily percentile ranks.

    All-stock ranks are computed BEFORE selecting the stock. Default stock is
    the alphabetically first code among stocks with maximum date coverage,
    chosen without using returns or model performance.
    """
    data = _prepare(predictions, portfolio_size)
    if security_code is None:
        coverage = data.groupby("SecuritiesCode")["Date"].nunique()
        security_code = sorted(
            coverage.index[coverage.eq(coverage.max())].tolist()
        )[0]
    history = data.loc[
        data["SecuritiesCode"].eq(str(security_code))
    ].sort_values("Date")
    if history.empty:
        raise ValueError(f"Security {security_code} is absent from predictions.")
    with plt.rc_context({"font.size": 10, "figure.facecolor": "white"}):
        fig, ax = plt.subplots(figsize=(13, 4.8))
        ax.plot(
            history["Date"], history["ActualPercentile"], color="#D47B28",
            alpha=0.45, linewidth=0.8, marker=".", markersize=3,
            label="Actual daily return rank",
        )
        ax.plot(
            history["Date"], history["PredictedPercentile"], color="#2266A7",
            linewidth=1.4, label="Predicted daily rank",
        )
        upper = 100 * (1 - portfolio_size / history["UniverseSize"])
        lower = 100 * portfolio_size / history["UniverseSize"]
        ax.plot(
            history["Date"], upper, "--", color="#718096", linewidth=0.8,
            label=f"Top/bottom {portfolio_size} cutoffs (approximate with ties)",
        )
        ax.plot(history["Date"], lower, "--", color="#718096", linewidth=0.8)
        ax.set(
            ylim=(0, 100), xlabel="Prediction date",
            ylabel="Daily percentile rank (higher = better)",
            title=f"{split_label} · stock {security_code} · ranking over time",
        )
        locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        ax.legend(frameon=False, loc="upper left", fontsize=9)
        _style(ax)
        fig.tight_layout()
    return fig, history.reset_index(drop=True)

In [ ]:


def _prepare(predictions, portfolio_size):
    required = {"Date", "SecuritiesCode", "Prediction", "Target"}
    missing = required.difference(predictions.columns)
    if missing:
        raise ValueError(f"Missing prediction columns: {sorted(missing)}")
    if not isinstance(portfolio_size, (int, np.integer)) or portfolio_size < 1:
        raise ValueError("portfolio_size must be a positive integer.")
    data = predictions.copy()
    if data.empty or data[["Date", "SecuritiesCode"]].isna().any().any():
        raise ValueError("Prediction data must be nonempty with valid dates/codes.")
    if pd.api.types.is_numeric_dtype(data["Date"]):
        raise ValueError("Parse numeric date codes explicitly before plotting.")
    data["Date"] = pd.to_datetime(data["Date"], errors="raise")
    data["SecuritiesCode"] = data["SecuritiesCode"].astype(str)
    if data.duplicated(["Date", "SecuritiesCode"]).any():
        raise ValueError("Select one model/split: duplicate stock-date rows found.")
    for column in ["Prediction", "Target"]:
        data[column] = pd.to_numeric(data[column], errors="raise")
        if not np.isfinite(data[column].to_numpy(dtype=float)).all():
            raise ValueError(f"{column} contains nonfinite values.")
    data = data.sort_values(
        ["Date", "Prediction", "SecuritiesCode"],
        ascending=[True, False, True], kind="mergesort",
    ).reset_index(drop=True)
    group = data.groupby("Date", sort=False)
    n = group["Target"].transform("size")
    if (n < 2 * portfolio_size).any():
        raise ValueError("Use full daily cross-sections with at least 2*K stocks.")
    if "UniverseSize" in data and not data["UniverseSize"].eq(n).all():
        raise ValueError("Prediction rows do not match the recorded universe size.")
    data["UniverseSize"] = n
    data["PredictedPosition"] = group.cumcount() + 1
    if "Rank" in data and not data["Rank"].eq(
        data["PredictedPosition"] - 1
    ).all():
        raise ValueError("Stored Rank differs from the evaluator's tie ordering.")
    for column, prefix in [("Prediction", "Predicted"), ("Target", "Actual")]:
        average_rank = group[column].rank(method="average", ascending=True)
        data[f"{prefix}Percentile"] = 100 * (average_rank - 0.5) / n
        if prefix == "Actual":
            data["ActualPosition"] = n + 1 - average_rank
    data["Side"] = np.select(
        [data["PredictedPosition"].le(portfolio_size),
         data["PredictedPosition"].gt(n - portfolio_size)],
        ["Long", "Short"], default="Middle",
    )
    return data


def _style(ax):
    ax.set_axisbelow(True)
    ax.grid(alpha=0.18)
    ax.spines[["top", "right"]].set_visible(False)


def plot_stock_rank_snapshot(
    predictions, date=None, *, portfolio_size=200,
    stocks_per_side=6, split_label="2021 test",
):
    """Show all stocks on one date plus an outcome-independent tail selection.

    Default date is the middle AVAILABLE date, not the best-performing date.
    Right panel selects the strongest predicted longs/shorts before looking at
    actual outcomes. Returned rows include their realized return and positions.
    """
    if not 1 <= stocks_per_side <= portfolio_size:
        raise ValueError("stocks_per_side must be between 1 and portfolio_size.")
    data = _prepare(predictions, portfolio_size)
    dates = data["Date"].drop_duplicates().sort_values()
    chosen_date = (
        dates.iloc[len(dates) // 2] if date is None else pd.Timestamp(date)
    )
    day = data.loc[data["Date"].eq(chosen_date)].copy()
    if day.empty:
        raise ValueError(f"No prediction rows on {chosen_date.date()}.")
    if day["Prediction"].nunique() < 2 or day["Target"].nunique() < 2:
        ic_text = "IC undefined (constant scores or targets)"
    else:
        ic = day["PredictedPercentile"].corr(day["ActualPercentile"])
        ic_text = f"Daily Spearman IC = {ic:.4f}"

    selected = pd.concat([
        day.loc[day["Side"].eq("Long")].head(stocks_per_side),
        day.loc[day["Side"].eq("Short")].tail(stocks_per_side),
    ]).sort_values("PredictedPosition")

    with plt.rc_context({"font.size": 10, "figure.facecolor": "white"}):
        fig, axes = plt.subplots(
            1, 2, figsize=(14, 6.2), gridspec_kw={"width_ratios": [1, 1.1]}
        )
        colors = {"Middle": "#C4CBD2", "Long": "#2266A7", "Short": "#D47B28"}
        for side in ["Middle", "Long", "Short"]:
            rows = day.loc[day["Side"].eq(side)]
            axes[0].scatter(
                rows["PredictedPercentile"], rows["ActualPercentile"],
                color=colors[side], s=10 if side == "Middle" else 20,
                alpha=0.3 if side == "Middle" else 0.65,
                label=side if side == "Middle" else f"{side} {portfolio_size}",
                edgecolors="none",
            )
        axes[0].plot([0, 100], [0, 100], "--", color="#38424D",
                     linewidth=1, label="Perfect ordering")
        axes[0].set(
            xlim=(0, 100), ylim=(0, 100),
            xlabel="Predicted percentile (higher = better)",
            ylabel="Actual return percentile (higher = better)",
            title=f"All {len(day):,} stocks on one date\n{ic_text}",
        )
        axes[0].legend(frameon=False, fontsize=9, loc="upper left")

        positions = np.arange(len(selected))
        axes[1].hlines(
            positions, selected["PredictedPercentile"],
            selected["ActualPercentile"], color="#ADB5BE", linewidth=1.8,
        )
        axes[1].scatter(
            selected["PredictedPercentile"], positions, color="#2266A7",
            marker="D", s=48, label="Predicted",
        )
        axes[1].scatter(
            selected["ActualPercentile"], positions, color="#D47B28",
            s=48, label="Actual",
        )
        labels = []
        for _, row in selected.iterrows():
            name = (
                f" · {str(row['Name'])[:20]}"
                if "Name" in selected and pd.notna(row["Name"]) else ""
            )
            labels.append(f"{row['SecuritiesCode']}{name} ({row['Side']})")
        axes[1].set(
            xlim=(-2, 102), yticks=positions, yticklabels=labels,
            xlabel="Rank percentile (higher = better)",
            title="Where did the selected stocks actually finish?",
        )
        axes[1].invert_yaxis()
        axes[1].legend(frameon=False, loc="lower right")
        for ax in axes:
            _style(ax)
        fig.suptitle(
            f"{split_label} · {chosen_date:%Y-%m-%d} · predicted vs actual ranks",
            fontsize=15, fontweight="bold",
        )
        fig.text(
            0.5, 0.025,
            "Ranks are computed using every eligible stock on this date. "
            "Percentiles preserve ties; portfolio positions break prediction ties "
            "by security code. This snapshot is descriptive.",
            ha="center", fontsize=9, color="#4E5967",
        )
        fig.tight_layout(rect=(0, 0.08, 1, 0.9))

    columns = [
        "Date", "SecuritiesCode", "Side", "PredictedPosition",
        "ActualPosition", "PredictedPercentile", "ActualPercentile", "Target",
    ]
    if "Name" in selected:
        columns.insert(2, "Name")
    return fig, selected[columns].reset_index(drop=True)


def plot_stock_rank_history(
    predictions, security_code=None, *, portfolio_size=200,
    split_label="2021 test",
):
    """Compare one stock's predicted/actual daily percentile ranks.

    All-stock ranks are computed BEFORE selecting the stock. Default stock is
    the alphabetically first code among stocks with maximum date coverage,
    chosen without using returns or model performance.
    """
    data = _prepare(predictions, portfolio_size)
    if security_code is None:
        coverage = data.groupby("SecuritiesCode")["Date"].nunique()
        security_code = sorted(
            coverage.index[coverage.eq(coverage.max())].tolist()
        )[0]
    history = data.loc[
        data["SecuritiesCode"].eq(str(security_code))
    ].sort_values("Date")
    if history.empty:
        raise ValueError(f"Security {security_code} is absent from predictions.")
    with plt.rc_context({"font.size": 10, "figure.facecolor": "white"}):
        fig, ax = plt.subplots(figsize=(13, 4.8))
        ax.plot(
            history["Date"], history["ActualPercentile"], color="#D47B28",
            alpha=0.45, linewidth=0.8, marker=".", markersize=3,
            label="Actual daily return rank",
        )
        ax.plot(
            history["Date"], history["PredictedPercentile"], color="#2266A7",
            linewidth=1.4, label="Predicted daily rank",
        )
        upper = 100 * (1 - portfolio_size / history["UniverseSize"])
        lower = 100 * portfolio_size / history["UniverseSize"]
        ax.plot(
            history["Date"], upper, "--", color="#718096", linewidth=0.8,
            label=f"Top/bottom {portfolio_size} cutoffs (approximate with ties)",
        )
        ax.plot(history["Date"], lower, "--", color="#718096", linewidth=0.8)
        ax.set(
            ylim=(0, 100), xlabel="Prediction date",
            ylabel="Daily percentile rank (higher = better)",
            title=f"{split_label} · stock {security_code} · ranking over time",
        )
        locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        ax.legend(frameon=False, loc="upper left", fontsize=9)
        _style(ax)
        fig.tight_layout()
    return fig, history.reset_index(drop=True)


def plot_model_rank_deciles(
    predictions_by_model, *, portfolio_size=200, split_label="2021 test",
):
    """Compare full-data, equal-date-weighted rank-decile returns.

    Input is a dict mapping model names to FULL prediction DataFrames. Comparisons
    require identical stock-date rows. Decile 1 is lowest predicted score and
    decile 10 highest. Score ties stay together and can leave uneven/missing bins.
    This descriptive plot has no significance or confidence-interval claims.
    """
    if not predictions_by_model:
        raise ValueError("Supply at least one model prediction table.")
    profiles = []
    reference_keys = None
    reference_targets = None
    for model_name, predictions in predictions_by_model.items():
        data = _prepare(predictions, portfolio_size)
        keys = pd.MultiIndex.from_frame(
            data[["Date", "SecuritiesCode"]].sort_values(
                ["Date", "SecuritiesCode"]
            )
        )
        if reference_keys is None:
            reference_keys = keys
            reference_targets = data.sort_values(
                ["Date", "SecuritiesCode"]
            )["Target"].to_numpy()
        elif not keys.equals(reference_keys):
            raise ValueError("Compare models on identical stock-date rows.")
        elif not np.array_equal(
            data.sort_values(["Date", "SecuritiesCode"])["Target"].to_numpy(),
            reference_targets,
        ):
            raise ValueError("Compared models must use identical actual targets.")
        data["Decile"] = np.ceil(
            data["PredictedPercentile"] / 10
        ).clip(1, 10).astype(int)
        daily = data.groupby(["Date", "Decile"], sort=True)["Target"].mean()
        profile = daily.groupby("Decile").agg(
            MeanActualReturn="mean", Dates="size"
        ).reindex(range(1, 11))
        profile["Model"] = model_name
        profiles.append(profile.reset_index())
    profile = pd.concat(profiles, ignore_index=True)

    with plt.rc_context({"font.size": 10, "figure.facecolor": "white"}):
        fig, ax = plt.subplots(figsize=(9, 5))
        for model_name, rows in profile.groupby("Model", sort=False):
            ax.plot(
                rows["Decile"], 100 * rows["MeanActualReturn"],
                marker="o", linewidth=1.8, label=model_name,
            )
        ax.axhline(0, color="#718096", linewidth=0.8)
        ax.set(
            xticks=range(1, 11),
            xlabel="Predicted score decile (1 lowest; 10 highest)",
            ylabel="Mean actual target return (%)",
            title=f"{split_label} · do higher scores identify higher returns?",
        )
        ax.legend(frameon=False)
        _style(ax)
        fig.text(
            0.5, 0.015,
            "Average within each date/decile, then across dates. Ties stay together. "
            "Bucket means differ from the weighted JPX spread score.",
            ha="center", fontsize=9, color="#4E5967",
        )
        fig.tight_layout(rect=(0, 0.06, 1, 1))
    return fig, profile



In [ ]:
fig, stock_history = plot_stock_rank_history(test_ranked)
plt.show()